# 03 – Feature Engineering

## Project Overview

In this notebook, I prepare the cleaned LinkedIn job postings dataset for machine learning.

The goal is to transform the raw feature columns into model-ready inputs by separating the target variable, splitting the data into training and testing sets, and defining preprocessing steps for categorical and numerical features.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Load data
project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

df = pd.read_csv(processed_dir / "clean_jobs.csv")

df.head()

,title,location,formatted_work_type,formatted_experience_level,company_name,views,normalized_salary
0,Marketing Coordinator,"Princeton, NJ",Full-time,Unknown,Corcoran Sawyer Smith,20.0,38480.0
1,Mental Health Therapist/Counselor,"Fort Collins, CO",Full-time,Unknown,Unknown,1.0,83200.0
2,Assitant Restaurant Manager,"Cincinnati, OH",Full-time,Unknown,The National Exemplar,8.0,55000.0
3,Senior Elder Law / Trusts and Estates Associat...,"New Hyde Park, NY",Full-time,Unknown,"Abrams Fensterman, LLP",16.0,157500.0
4,Service Technician,"Burlington, IA",Full-time,Unknown,Unknown,3.0,70000.0


## Separate Features and Target

The target variable is `normalized_salary`, which represents the annual salary I want the model to predict.

All remaining columns are used as input features.

In [3]:
X = df.drop(columns=["normalized_salary"])
y = df["normalized_salary"]

X.head()

,title,location,formatted_work_type,formatted_experience_level,company_name,views
0,Marketing Coordinator,"Princeton, NJ",Full-time,Unknown,Corcoran Sawyer Smith,20.0
1,Mental Health Therapist/Counselor,"Fort Collins, CO",Full-time,Unknown,Unknown,1.0
2,Assitant Restaurant Manager,"Cincinnati, OH",Full-time,Unknown,The National Exemplar,8.0
3,Senior Elder Law / Trusts and Estates Associat...,"New Hyde Park, NY",Full-time,Unknown,"Abrams Fensterman, LLP",16.0
4,Service Technician,"Burlington, IA",Full-time,Unknown,Unknown,3.0


In [4]:
y.head()

0     38480.0
1     83200.0
2     55000.0
3    157500.0
4     70000.0
Name: normalized_salary, dtype: float64

## Identify Feature Types

The dataset contains both categorical and numerical features.

Categorical features will need to be encoded before modeling, while numerical features can be scaled.

In [5]:
categorical_features = [
    "title",
    "location",
    "formatted_work_type",
    "formatted_experience_level",
    "company_name"
]

numeric_features = [
    "views"
]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Categorical features: ['title', 'location', 'formatted_work_type', 'formatted_experience_level', 'company_name']
Numeric features: ['views']


## Train-Test Split

I split the dataset into training and testing sets so that model performance can be evaluated on unseen data.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Features:", X_train.shape)
print("Testing Features :", X_test.shape)
print("Training Target  :", y_train.shape)
print("Testing Target   :", y_test.shape)

Training Features: (28448, 6)
Testing Features : (7112, 6)
Training Target  : (28448,)
Testing Target   : (7112,)


## Build a Preprocessing Pipeline

Machine learning models require numerical input features. Since this dataset contains both categorical and numerical variables, I create a preprocessing pipeline to transform the data into a format suitable for modeling.

The preprocessing pipeline performs the following operations:

- One-hot encodes categorical variables
- Standardizes numerical variables
- Applies the same transformations consistently to both the training and testing datasets

Using a preprocessing pipeline helps prevent data leakage and ensures that future datasets are processed in exactly the same way as the training data.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numeric_features
        )
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

## Apply the Preprocessing Pipeline

I fit the preprocessing pipeline only on the training data, then apply the learned transformations to the testing data.

This prevents data leakage because the test set should remain unseen until model evaluation. The training data is used to learn the encoding structure and scaling parameters, while the test data is transformed using those same learned rules.

In [8]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [9]:
print(type(X_train_processed))

print(X_train_processed.shape)

print(X_test_processed.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(28448, 32774)
(7112, 32774)


## Inspect Engineered Features

After preprocessing, I inspect the generated feature names to understand how the original variables were transformed.

This helps confirm that categorical variables were expanded into one-hot encoded features and that the numerical feature was included in the final feature matrix.

In [10]:
feature_names = preprocessor.get_feature_names_out()

print("Number of engineered features:", len(feature_names))

feature_names[:20]

Number of engineered features: 32774


array(['categorical__title_#13635 - Salesforce Manual Tester',
       'categorical__title_#13644 - Data Collection Associate',
       'categorical__title_#227 Staff Pharmacist - $10,000 Sign-on Bonus',
       'categorical__title_$170k-$210k – IAM Engineer',
       'categorical__title_$54 / hour - Home Health MSW',
       'categorical__title_(Entry Level) Sales Executive',
       'categorical__title_(MA) Licensed Professional Counselor/Licensed Mental Health Counselor (Contract)',
       'categorical__title_(NY) Licensed Marriage and Family Therapist (Contract)',
       'categorical__title_(Onsite) Retail Banker',
       'categorical__title_(RN) Clinical Case Manager',
       'categorical__title_(Remote) Inside Sales Representative (Uncapped Commission + Base Pay & No Cold Calling)',
       'categorical__title_(SAP) IT Business Systems Manager',
       'categorical__title_(Senior) Bioinformatician, Antibody Engineering',
       'categorical__title_(Senior) Vice President of Lending',
  

### Feature Engineering Result

After standardizing text fields and applying one-hot encoding, the original 6 input features expanded into **32,774 engineered features**.

This increase is expected because high-cardinality categorical variables such as job title, location, and company name contain many unique values. One-hot encoding converts each unique category into its own binary indicator feature.

The processed data is stored as a sparse matrix, which is memory-efficient because most one-hot encoded values are zero.

In [11]:
print("Training observations:", X_train_processed.shape[0])
print("Testing observations :", X_test_processed.shape[0])
print("Engineered features  :", X_train_processed.shape[1])

Training observations: 28448
Testing observations : 7112
Engineered features  : 32774


# Summary

In this notebook, I transformed the cleaned LinkedIn job postings dataset into a machine-learning-ready feature matrix.

### Completed Tasks

- Loaded the cleaned dataset
- Separated the input features and target variable
- Identified categorical and numerical features
- Split the dataset into training and testing sets
- Built a preprocessing pipeline using `ColumnTransformer`
- One-hot encoded categorical variables
- Standardized the numerical feature
- Generated a sparse feature matrix containing **32,774 engineered features**

The processed feature matrix is now ready for training and evaluating machine learning models in the next notebook.